In [ ]:
# ==================================================
# Experimento 3: Simulación Monte Carlo
# ==================================================

# ¿Por qué eliminamos el D original en cada simulación?
# Si NO eliminamos el D original, todas las 1000 simulaciones usarían
# exactamente la misma asignación de tratamiento y darían el mismo resultado.
# El Monte Carlo no tendría sentido.
#
# Lo que queremos es simular "¿qué pasaría si repetimos el estudio 1000 veces?":
# - En cada simulación mantenemos los mismos resultados potenciales (yd0, yd1)
# - Pero re-asignamos el tratamiento de forma diferente

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar datos de clase
url = "https://raw.githubusercontent.com/adiazescobar/libro_cortes/main/dofile/04_ParametrosStata/04_data.dta"
df_clase = pd.read_stata(url)

# Expandir a 80,000 observaciones
df_expandido = pd.concat([df_clase]*10000, ignore_index=True)

# --------------------------
# Escenario 1: Con SELECCIÓN (viola independencia)
# --------------------------

print("\n=== MONTE CARLO: Escenario 1 - CON SELECCIÓN ===")

np.random.seed(12345)
n_sims = 1000

# Array para almacenar resultados
SESGO_sel = np.zeros(n_sims)

# Loop de Monte Carlo
for i in range(n_sims):

    # Usar datos expandidos de clase
    df_sim = df_expandido.copy()

    # IMPORTANTE: Eliminamos el D original y creamos uno nuevo en cada simulación
    # Si no hacemos esto, todas las simulaciones darían el mismo resultado

    # SELECCIÓN: Los que tienen mejor yd0 se tratan más
    mean_yd0 = df_sim["yd0"].mean()
    prob_D = 1 / (1 + np.exp(-(df_sim["yd0"] - mean_yd0)/2))  # Mayor yd0 → mayor prob de D=1
    D = (np.random.rand(len(df_sim)) < prob_D).astype(int)

    # Generar resultado observado y efecto individual
    y = D * df_sim["yd1"] + (1 - D) * df_sim["yd0"]
    tau = df_sim["yd1"] - df_sim["yd0"]

    # Calcular estimadores (misma nomenclatura que en clase)
    ATE = tau.mean()
    ATT = tau[D == 1].mean()
    ybar_1 = y[D == 1].mean()
    ybar_0 = y[D == 0].mean()
    NAIVE = ybar_1 - ybar_0

    # Guardar el sesgo de esta simulación
    # Recordar: NAIVE = ATT + SESGO, por lo tanto SESGO = NAIVE - ATT
    SESGO_sel[i] = NAIVE - ATT

    # Mostrar progreso
    if (i + 1) % 100 == 0:
        print(f"Simulación {i+1} de {n_sims} completada")

# Resultados
print("\n=== RESULTADOS CON SELECCIÓN (viola independencia) ===")
print(f"Sesgo promedio del estimador Naive: {SESGO_sel.mean():.4f}")
print("El sesgo persiste incluso con muchas observaciones!")

# Gráfico
plt.figure(figsize=(10, 6))
plt.hist(SESGO_sel, bins=30, color='red', alpha=0.5, edgecolor='black')
plt.axvline(0, color='darkred', linewidth=2, label='Sesgo cero (ideal)')
plt.xlabel('SESGO = NAIVE - ATT')
plt.ylabel('Frecuencia')
plt.title('Distribución del Sesgo del Estimador Naive\n1000 simulaciones con SELECCIÓN - Datos de clase')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('sesgo_con_seleccion.png', dpi=300, bbox_inches='tight')
plt.show()

# --------------------------
# Escenario 2: Con ALEATORIZACIÓN (cumple independencia)
# --------------------------

print("\n=== MONTE CARLO: Escenario 2 - CON ALEATORIZACIÓN ===")

np.random.seed(12345)

# Array para almacenar resultados
SESGO_aleat = np.zeros(n_sims)

# Loop de Monte Carlo
for i in range(n_sims):

    # Usar datos expandidos de clase
    df_sim = df_expandido.copy()

    # IMPORTANTE: Eliminamos el D original y creamos uno nuevo en cada simulación
    # Si no hacemos esto, todas las simulaciones darían el mismo resultado

    # ALEATORIZACIÓN: D es independiente de yd0 y yd1
    D = (np.random.rand(len(df_sim)) < 0.5).astype(int)  # 50% tratamiento, 50% control

    # Generar resultado observado y efecto individual
    y = D * df_sim["yd1"] + (1 - D) * df_sim["yd0"]
    tau = df_sim["yd1"] - df_sim["yd0"]

    # Calcular estimadores (misma nomenclatura que en clase)
    ATE = tau.mean()
    ATT = tau[D == 1].mean()
    ybar_1 = y[D == 1].mean()
    ybar_0 = y[D == 0].mean()
    NAIVE = ybar_1 - ybar_0

    # Guardar el sesgo de esta simulación
    # Recordar: NAIVE = ATT + SESGO, por lo tanto SESGO = NAIVE - ATT
    SESGO_aleat[i] = NAIVE - ATT

    # Mostrar progreso
    if (i + 1) % 100 == 0:
        print(f"Simulación {i+1} de {n_sims} completada")

# Resultados
print("\n=== RESULTADOS CON ALEATORIZACIÓN (cumple independencia) ===")
print(f"Sesgo promedio del estimador Naive: {SESGO_aleat.mean():.4f}")
print("El sesgo es aproximadamente CERO!")

# Gráfico
plt.figure(figsize=(10, 6))
plt.hist(SESGO_aleat, bins=30, color='green', alpha=0.5, edgecolor='black')
plt.axvline(0, color='darkgreen', linewidth=2, label='Sesgo cero')
plt.xlabel('SESGO = NAIVE - ATT')
plt.ylabel('Frecuencia')
plt.title('Distribución del Sesgo del Estimador Naive\n1000 simulaciones con ALEATORIZACIÓN - Datos de clase')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('sesgo_con_aleatorizacion.png', dpi=300, bbox_inches='tight')
plt.show()

# --------------------------
# Comparación lado a lado
# --------------------------

plt.figure(figsize=(12, 6))
plt.hist(SESGO_sel, bins=30, color='red', alpha=0.4, label='Con selección', edgecolor='black')
plt.hist(SESGO_aleat, bins=30, color='green', alpha=0.4, label='Con aleatorización', edgecolor='black')
plt.axvline(0, color='black', linewidth=2, linestyle='--')
plt.xlabel('SESGO = NAIVE - ATT')
plt.ylabel('Frecuencia')
plt.title('Comparación: Sesgo con vs sin aleatorización\n1000 simulaciones Monte Carlo - Datos de clase')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('comparacion_monte_carlo.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== FIN DE LA SIMULACIÓN MONTE CARLO ===")

In [ ]:
# Clase 4 - Estimadores Causales en Secciones Transversales
# Profesora: Ana Díaz

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# --------------------------
# Cargar datos
# --------------------------
url = "https://raw.githubusercontent.com/adiazescobar/libro_cortes/main/dofile/04_ParametrosStata/04_data.dta"
df = pd.read_stata(url)

# Generar resultado observado
df["y"] = df["D"] * df["yd1"] + (1 - df["D"]) * df["yd0"]

# --------------------------
# Estadísticas descriptivas
# --------------------------
print(df["D"].value_counts())
print(df["y"].describe())
print(df.groupby("D")["y"].agg(["mean", "std"]))

# --------------------------
# Diferencia de medias (t-test)
# --------------------------
treated = df[df["D"] == 1]["y"]
control = df[df["D"] == 0]["y"]
t_stat, p_val = stats.ttest_ind(treated, control)
print("t-test:", t_stat, "p-value:", p_val)

# --------------------------
# Regresión simple
# --------------------------
X = sm.add_constant(df["D"])
model = sm.OLS(df["y"], X).fit(cov_type='HC1')  # robust SE
print(model.summary())

# --------------------------
# Estimadores causales
# --------------------------
df["tau"] = df["yd1"] - df["yd0"]

def estimadores(tau, y, D):
    ATE = tau.mean()
    ATT = tau[D == 1].mean()
    ATU = tau[D == 0].mean()
    ybar_1 = y[D == 1].mean()
    ybar_0 = y[D == 0].mean()
    NAIVE = ybar_1 - ybar_0
    print("--- Estimadores ---")
    print("ATE =", ATE)
    print("ATT =", ATT)
    print("ATU =", ATU)
    print("Naive =", NAIVE)
    print("Sesgo de Selección =", NAIVE - ATT)

estimadores(df["tau"], df["y"], df["D"])

# --------------------------
# Experimento 1: Aumentar muestra
# --------------------------
df_expanded = pd.concat([df]*10000, ignore_index=True)
df_expanded["y"] = df_expanded["D"] * df_expanded["yd1"] + (1 - df_expanded["D"]) * df_expanded["yd0"]
df_expanded["tau"] = df_expanded["yd1"] - df_expanded["yd0"]
estimadores(df_expanded["tau"], df_expanded["y"], df_expanded["D"])

# --------------------------
# Experimento 2: Asignación aleatoria
# --------------------------
np.random.seed(87634)
df_random = df.copy()
df_random["D"] = (np.random.rand(len(df_random)) > 0.5).astype(int)
df_random["y"] = df_random["D"] * df_random["yd1"] + (1 - df_random["D"]) * df_random["yd0"]
df_random["tau"] = df_random["yd1"] - df_random["yd0"]
estimadores(df_random["tau"], df_random["y"], df_random["D"])
